Link: [Ultimate RAG Bootcamp Using Langchain,LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

# Query Enhancement – Query Expansion Techniques

## 📚 Learning Objectives
By the end of this notebook, you will understand:
1. What query expansion is and why it's important in RAG systems
2. How to use an LLM to automatically expand user queries
3. How to integrate query expansion into a complete RAG pipeline

---

## 🎯 What is Query Enhancement?

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM's final answer will be.

**Query enhancement** refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.

### When is Query Expansion Useful?

| Scenario | Problem | How Expansion Helps |
|----------|---------|---------------------|
| Short queries | "memory" lacks context | Adds related terms: "memory management, RAM, storage" |
| Ambiguous queries | "apple" - fruit or company? | Adds domain context based on user intent |
| Technical jargon mismatch | User says "ML" but docs say "machine learning" | Bridges terminology gaps |
| Spelling variants | "colour" vs "color" | Captures all variants |

### 🔄 How Query Expansion Works

```
User Query: "LangChain memory"
        ↓
   [LLM Expansion]
        ↓
Expanded Query: "LangChain memory modules ConversationBufferMemory 
                 ConversationSummaryMemory context retention state management"
        ↓
   [Better Retrieval]
        ↓
   More Relevant Documentssssssss

### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [ ]:
# ============================================================================
# STEP 0: Import Required Libraries
# ============================================================================
# 
# We need several components from LangChain:
# - Document loading and splitting utilities
# - Embedding models for semantic search
# - Vector store for efficient similarity search
# - LLM and prompting utilities
# - Chain components for building the RAG pipeline
# ============================================================================

from langchain.document_loaders import TextLoader           # Load text files
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Split text into chunks
from langchain_huggingface import HuggingFaceEmbeddings     # Free, local embeddings
from langchain_community.vectorstores import FAISS          # Fast similarity search
from langchain.chat_models import init_chat_model           # Initialize LLM
from langchain.prompts import PromptTemplate                # Create prompts
from langchain.chains.combine_documents import create_stuff_documents_chain  # Combine docs
from langchain.chains.retrieval import create_retrieval_chain  # Retrieval chain
from langchain_core.output_parsers import StrOutputParser   # Parse LLM output as string
from langchain_core.runnables import RunnableMap            # Parallel execution

In [ ]:
# ============================================================================
# STEP 1: Load and Split the Dataset
# ============================================================================
# 
# We load a text file containing information about LangChain and CrewAI.
# The text is then split into smaller chunks for efficient retrieval.
#
# Key Parameters:
# - chunk_size=300: Each chunk contains ~300 characters (roughly 50-75 words)
# - chunk_overlap=50: Adjacent chunks share 50 characters for context continuity
#
# 💡 TIP: Smaller chunks are better for precise retrieval but may lose context.
#         Larger chunks retain more context but may include irrelevant information.
# ============================================================================

loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()

# RecursiveCharacterTextSplitter tries to split at natural boundaries
# (paragraphs, sentences, words) before resorting to character-level splits
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

print(f"📄 Loaded document and created {len(chunks)} chunks")


In [ ]:
# Let's inspect the chunks to understand our data
# Each chunk is a Document object with 'page_content' and 'metadata'
print(f"Sample chunk content:\n{chunks[0].page_content}\n")
print(f"Chunk metadata: {chunks[0].metadata}")
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [ ]:
# ============================================================================
# STEP 2: Create Vector Store with Embeddings
# ============================================================================
# 
# We use HuggingFace's all-MiniLM-L6-v2 model for embeddings:
# - Free and runs locally (no API costs)
# - Produces 384-dimensional vectors
# - Good balance of speed and quality
#
# FAISS (Facebook AI Similarity Search) is used for the vector store:
# - Extremely fast similarity search
# - Supports various index types for different use cases
# - Works well for datasets up to millions of vectors
# ============================================================================

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"✅ Created vector store with {len(chunks)} vectors")

# ============================================================================
# STEP 3: Create MMR Retriever
# ============================================================================
# 
# MMR (Maximal Marginal Relevance) balances:
# - Relevance: How similar is the document to the query?
# - Diversity: How different is this document from already selected ones?
#
# This prevents retrieving multiple very similar documents,
# ensuring we get a diverse set of relevant information.
# ============================================================================

retriever = vectorstore.as_retriever(
    search_type="mmr",      # Use MMR for diverse results
    search_kwargs={"k": 5}  # Return top 5 documents
)
print("✅ Created MMR retriever (k=5)")
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x358ac1610>, search_type='mmr', search_kwargs={'k': 5})

In [ ]:
# ============================================================================
# STEP 4: Initialize the LLM
# ============================================================================
# 
# We use OpenAI's o4-mini model for two purposes:
# 1. Query Expansion: Generating expanded versions of user queries
# 2. Answer Generation: Producing final answers based on retrieved context
#
# 💡 TIP: You can swap this with other models like:
#    - "groq:llama-3.1-8b-instant" (fast, free tier available)
#    - "openai:gpt-4o" (more capable, higher cost)
#    - "anthropic:claude-3-haiku" (fast and affordable)
# ============================================================================

import os
from dotenv import load_dotenv
load_dotenv()

# Set the API key from environment variables
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Initialize the LLM
llm = init_chat_model("openai:o4-mini")
print("✅ LLM initialized successfully")
llm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x3b4b25750>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x3b48ac790>, root_client=<openai.OpenAI object at 0x3b4867a50>, root_async_client=<openai.AsyncOpenAI object at 0x3b4fc2ed0>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [ ]:
# ============================================================================
# STEP 5: Create the Query Expansion Chain
# ============================================================================
# 
# This is the CORE of query expansion! We create a prompt that instructs
# the LLM to enrich the user's original query with:
# 
# 1. Synonyms: Different words with similar meanings
# 2. Technical terms: Domain-specific vocabulary
# 3. Related concepts: Topics that often appear together
# 4. Context: Additional information about what the user might want
#
# The chain uses LCEL (LangChain Expression Language):
#   prompt | llm | output_parser
#
# This creates a pipeline where:
#   1. The prompt formats the input
#   2. The LLM generates the expanded query
#   3. The StrOutputParser extracts just the text response
# ============================================================================

query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

# Create the chain using LCEL (LangChain Expression Language)
# This pipes the prompt → LLM → output parser together
query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()

print("✅ Query expansion chain created")
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x3b4b25750>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x3b48ac790>, root_client=<openai.OpenAI object at 0x3b4867a50>, root_async_client=<openai.AsyncOpenAI object at 0x3b4fc2ed0>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
# ============================================================================
# 🧪 TEST: See Query Expansion in Action
# ============================================================================
# 
# Let's test our query expansion chain with a simple query.
# Notice how the LLM adds related terms, synonyms, and context!
# ============================================================================

original_query = "Langchain memory"
expanded_query = query_expansion_chain.invoke({"query": original_query})

print(f"📝 Original Query: {original_query}")
print(f"\n📈 Expanded Query:\n{expanded_query}")

'Here’s an example of a much more detailed search query that covers synonyms, related technical terms, module names, integrations and use-cases:\n\n“LangChain memory” OR “LangChain memory management” OR “LangChain context management” OR “LangChain state persistence”  \nAND (“ConversationBufferMemory” OR “ConversationSummaryMemory” OR “CombinedMemory” OR “RetrievalMemory” OR “ChatMemory” OR “VectorStoreRetrieverMemory”)  \nAND (“conversational memory” OR “session memory” OR “short-term memory” OR “long-term memory” OR “context buffer” OR “summary memory” OR “knowledge base” OR “cache”)  \nAND (“vector store” OR “embedding store” OR “Redis” OR “FAISS” OR “Pinecone” OR “Weaviate” OR “PGVector”)  \nAND (“Python” OR “JavaScript” OR “Node.js” OR “best practices” OR “architecture” OR “implementation guide” OR “technical documentation” OR “GitHub examples”)  \n\nYou can paste that whole string into your search engine or split parts of it as needed.'

In [ ]:
# ============================================================================
# STEP 6: Create the Answer Generation Chain
# ============================================================================
# 
# This chain takes the retrieved documents (context) and the user's question,
# then generates a coherent answer.
#
# "Stuff" in create_stuff_documents_chain means we "stuff" all documents
# into the prompt context. This is the simplest approach, suitable when:
# - Total context fits within the model's context window
# - You have relatively few retrieved documents
#
# 💡 Alternative strategies for large contexts:
# - MapReduce: Process documents in batches, then combine
# - Refine: Iteratively improve the answer with each document
# ============================================================================

answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

# Create the document chain that "stuffs" all documents into context
document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)
print("✅ Answer generation chain created")

In [ ]:
# ============================================================================
# STEP 7: Build the Complete RAG Pipeline with Query Expansion
# ============================================================================
# 
# This is where everything comes together! The pipeline:
#
# 1. Takes the user's input query
# 2. Expands the query using the LLM (query_expansion_chain)
# 3. Retrieves relevant documents using the expanded query
# 4. Generates an answer using the retrieved context
#
# RunnableMap allows us to run multiple operations in parallel:
# - "input": passes through the original question (for the answer prompt)
# - "context": expands query → retrieves documents
#
# Architecture:
# ┌─────────────────────────────────────────────────────────────┐
# │                    RAG Pipeline                              │
# │                                                              │
# │  User Query ──┬── [Pass Through] ─────────── input ──┐      │
# │               │                                       │      │
# │               └── [Query Expansion] ── [Retriever] ── context│
# │                                                       │      │
# │                                            ↓          │      │
# │                                    [Document Chain] ←─┘      │
# │                                            ↓                 │
# │                                      Final Answer            │
# └─────────────────────────────────────────────────────────────┘
# ============================================================================

rag_pipeline = (
    RunnableMap({
        # Pass the original question through to the answer prompt
        "input": lambda x: x["input"],
        # Expand the query, then retrieve documents based on expanded query
        "context": lambda x: retriever.invoke(
            query_expansion_chain.invoke({"query": x["input"]})
        )
    })
    | document_chain  # Generate answer from context + question
)

print("✅ Complete RAG pipeline with query expansion is ready!")

In [ ]:
# ============================================================================
# STEP 8: Run the Pipeline - Example 1
# ============================================================================
# 
# Let's test our query expansion RAG pipeline!
# We'll first show the expanded query, then the final answer.
# ============================================================================

query = {"input": "What types of memory does LangChain support?"}

# First, let's see how the query gets expanded
print("📝 Original Query:", query["input"])
print("\n📈 Expanded Query:")
print(query_expansion_chain.invoke({"query": query["input"]}))

# Now run the full pipeline
print("\n" + "="*60)
response = rag_pipeline.invoke(query)
print("✅ Final Answer:\n", response)

Expanded query:  
“What types of memory does LangChain support, including all built-in memory modules (for example BufferMemory, ConversationBufferMemory, ConversationSummaryMemory, SummaryMemory, EntityMemory, KnowledgeGraphMemory, VectorStoreMemory, CombinedMemory), and what short-term versus long-term memory strategies does it offer? Please include details on memory backends and integrations (Redis, Chroma, FAISS, Pinecone, Supabase, SQLite, in-memory stores), conversational state persistence, context window management, session memory, buffer versus summary memory, and any relevant technical terms or synonyms (stateful memory, context retention, conversation history storage) used in the LangChain Python SDK.”
✅ Answer:
 LangChain currently ships with two out-of-the-box memory implementations:  
1. ConversationBufferMemory  
   • Keeps the full raw history of the dialogue in memory  
   • Lets the model “see” every prior turn (up to your token limits)  
2. ConversationSummaryMemory  

In [ ]:
# ============================================================================
# STEP 9: Run the Pipeline - Example 2
# ============================================================================
# 
# Let's try a different query about CrewAI agents.
# Notice how query expansion helps even with short, vague queries!
# ============================================================================

query = {"input": "CrewAI agents?"}

# Show the expanded query
print("📝 Original Query:", query["input"])
print("\n📈 Expanded Query:")
print(query_expansion_chain.invoke({"query": query["input"]}))

# Run the full pipeline
print("\n" + "="*60)
response = rag_pipeline.invoke(query)
print("✅ Final Answer:\n", response)

Expanded query:

("CrewAI agents" OR "Crew AI agents" OR "CrewAI bots" OR "Crew AI assistants" OR "autonomous crew agents" OR "AI-driven crew management assistants" OR "virtual crew members" OR "digital crew agents")  
AND  
("crew scheduling" OR "workforce management" OR "staff rostering" OR "resource allocation" OR "employee roster optimization")  
AND  
("multi-agent system" OR "autonomous agents" OR "distributed AI" OR "agent-based modeling")  
AND  
("machine learning" OR "reinforcement learning" OR "predictive analytics" OR "optimization algorithms" OR "real-time planning")  
AND  
("airline" OR "maritime" OR "hospitality" OR "logistics" OR "field service")
✅ Answer:
 CrewAI agents are semi-autonomous, role-specialized AI “workers” that team up in a predefined workflow to tackle complex, multi-step tasks.  Key points:  
1. Defined Roles  
   • Researcher – gathers data and insights  
   • Planner – lays out strategy, timelines, and dependencies  
   • Executor – carries out concr

## 📊 Summary: Query Expansion Benefits

| Aspect | Without Query Expansion | With Query Expansion |
|--------|------------------------|---------------------|
| Query "LangChain memory" | Matches only exact term | Matches: memory modules, ConversationBuffer, state management, etc. |
| Retrieval Coverage | Limited to exact matches | Broader, more comprehensive |
| Answer Quality | May miss relevant info | More complete answers |

## 🎯 Key Takeaways

1. **Query expansion uses an LLM** to enrich short or ambiguous queries
2. **Synonyms and related terms** help bridge vocabulary gaps
3. **The expanded query goes to the retriever**, not the original
4. **Best for**: Short queries, domain-specific jargon, ambiguous terms

## ⚠️ Considerations

- **Latency**: Adds one LLM call before retrieval
- **Cost**: Extra API calls increase costs
- **Over-expansion**: Too many terms can dilute relevance

## 🔗 Related Techniques

- **Query Decomposition**: Break complex queries into sub-questions
- **HyDE**: Generate hypothetical answers to improve retrieval
- **Query Rewriting**: Reformulate queries for clarity
